In [39]:
import pandas as pd
df = pd.read_csv("verlan_database.csv")
# only takes the words included in the wiktionary_verlan_database.csv
screen = pd.read_csv("wiktionary_verlan_titles.csv")

title_set = set(screen["title"].dropna().astype(str).str.strip().str.lower())
df = df[df["verlan_form"].astype(str).str.strip().str.lower().isin(title_set)].copy()

df_two_cols = df[["verlan_form", "base_form"]]

print(df_two_cols.head(100))
print(f"Rows after screen filter: {len(df)}")

   verlan_form      base_form
0   anti rebeu     antiarabe 
1       babtou         toubab
2   balle-peau  peau de balle
3    ballepeau  peau de balle
4      balpeau  peau de balle
..         ...            ...
95      keubla          black
96        keuf           flic
97        keum            mec
98       keumé            mec
99      keupon           punk

[100 rows x 2 columns]
Rows after screen filter: 290


## jellyfish

In [40]:
import jellyfish
for i, row in df.iterrows():
    w1 = row["verlan_form"]
    w2 = row["base_form"]
    if not isinstance(w1, str) or not isinstance(w2, str):
        print("not valid strings: ", i, w1, type(w1), w2, type(w2))
lev_dist = []
dam_dist = []
jaro_sim = []
for index, row in df_two_cols.iterrows():
    w1 = row["verlan_form"]
    w2 = row["base_form"]
    lev_dist.append(jellyfish.levenshtein_distance(w1, w2))
    dam_dist.append(jellyfish.damerau_levenshtein_distance(w1, w2))
    jaro_sim.append(jellyfish.jaro_similarity(w1, w2))
    # soundex1 = jellyfish.soundex(w1)
    # soundex2 = jellyfish.soundex(w2)
    # print(soundex1)
    # print(soundex2)
print("Levenshtein distance: ", sum(lev_dist) / len(lev_dist))
print("Damerau-Levenshtein distance: ", sum(dam_dist) / len(dam_dist))
print("Jaro similarity: ", sum(jaro_sim) / len(jaro_sim))

Levenshtein distance:  5.103448275862069
Damerau-Levenshtein distance:  5.1
Jaro similarity:  0.45886157463122895


## Spacy

In [41]:
import spacy
nlp = spacy.load("fr_core_news_lg")
similarities = []
for index, row in df_two_cols.iterrows():
    w1 = row["verlan_form"]
    w2 = row["base_form"]
    token1 = nlp(w1)
    token2 = nlp(w2)
    sim = token1.similarity(token2)
    similarities.append(sim)
    # print(w1, w2, sim)
print("Average similarity: ", sum(similarities) / len(similarities))
similarities_screen = [sim for sim in similarities if sim > 0]
print("Average similarity (screened): ", sum(similarities_screen) / len(similarities_screen))

/tmp/ipykernel_22042/748199791.py:9: UserWarning: [W008] Evaluating Doc.similarity based on empty vectors.
  sim = token1.similarity(token2)


Average similarity:  0.08325725994173212
Average similarity (screened):  0.2658699440240349


In [42]:
import spacy

nlp = spacy.load("fr_core_news_lg")   # 或 lg
print(nlp.meta["name"])

for w in ["meuf", "femme", "chelou", "louche", "rebeu", "arabe"]:
    doc = nlp(w)
    tok = doc[0]
    print(
        w,
        "has_vector=", tok.has_vector,
        "vector_norm=", tok.vector_norm,
        "is_oov=", tok.is_oov
    )

core_news_lg
meuf has_vector= True vector_norm= 23.746458 is_oov= False
femme has_vector= True vector_norm= 34.48047 is_oov= False
chelou has_vector= True vector_norm= 14.017049 is_oov= False
louche has_vector= True vector_norm= 17.562468 is_oov= False
rebeu has_vector= True vector_norm= 24.750566 is_oov= False
arabe has_vector= True vector_norm= 30.308107 is_oov= False


## Fasttext

In [43]:
import fasttext
import fasttext.util
fasttext.util.download_model('fr', if_exists='ignore')
ft = fasttext.load_model('cc.fr.300.bin')



In [44]:
vec = ft.get_word_vector("meuf")
print(vec.shape)
print(vec[:10])

(300,)
[-0.07452467  0.00135089  0.0236647   0.01450148 -0.07200458 -0.000401
  0.16172828  0.03869662 -0.06638871  0.01467166]


In [47]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

w1 = ft.get_word_vector("meuf")
w2 = ft.get_word_vector("femme")

sim = cosine_similarity(w1, w2)
print(sim)
print(ft.get_nearest_neighbors("meuf", k=10))

0.53607875
[(0.7963905930519104, 'nana'), (0.781146228313446, 'gonzesse'), (0.760162889957428, 'pétasse'), (0.7276779413223267, 'mec'), (0.7177290916442871, 'bonnasse'), (0.707476019859314, 'meufs'), (0.7058149576187134, 'pouffiasse'), (0.6935135126113892, 'petasse'), (0.6842332482337952, 'grognasse'), (0.6840963959693909, 'connasse')]


In [49]:
import pandas as pd
import numpy as np

df["fasttext_similarity"] = df.apply(
    lambda row: cosine_similarity(
        ft.get_word_vector(row["verlan_form"]),
        ft.get_word_vector(row["base_form"])
    ),
    axis=1
)
df["fasttext_difference"] = df.apply(
    lambda row: np.linalg.norm(
        ft.get_word_vector(row["verlan_form"]) - ft.get_word_vector(row["base_form"])
    ),
    axis=1
)


print(df[["verlan_form", "base_form", "fasttext_similarity"]].head(10))
print("Average similarity:", df["fasttext_similarity"].mean())
print("Average difference:", df["fasttext_difference"].mean())

  verlan_form      base_form  fasttext_similarity
0  anti rebeu     antiarabe              0.333782
1      babtou         toubab             0.476317
2  balle-peau  peau de balle             0.446452
3   ballepeau  peau de balle             0.465137
4     balpeau  peau de balle             0.335395
5       barje         jobard             0.360039
6       barjo         jobard             0.416660
7      barjot         jobard             0.508903
8    barjoter         jobard             0.241804
9       bebar          barbe             0.279475
Average similarity: 0.2619798
Average difference: 1.1564527


In [ ]:
print(ft.get_nearest_neighbors("meuf", k=10))

[(0.7963905930519104, 'nana'), (0.781146228313446, 'gonzesse'), (0.760162889957428, 'pétasse'), (0.7276779413223267, 'mec'), (0.7177290916442871, 'bonnasse'), (0.707476019859314, 'meufs'), (0.7058149576187134, 'pouffiasse'), (0.6935135126113892, 'petasse'), (0.6842332482337952, 'grognasse'), (0.6840963959693909, 'connasse')]
